# pool_v2_200000 분포 검증 — 블록별 시각화

생성 코드(`sampling/build_pool.py`)는 **effort(총 활성량)** 와 **direction(희소도, Dirichlet α)** 를
분리해서 샘플링하고, 7개 구조 블록을 쌓습니다:

| 블록 | 역할 | 분포 특징 |
|---|---|---|
| REST | 기준 pose | 원점(all-zero) 1개 |
| SINGLE | 단일근 sweep (dM/daᵢ 기저) | 축 위, n_active=1 |
| PAIR | 2근 격자 (2차 상호작용) | 좌표평면, n_active=2 |
| TRIPLE | 희소 3근 랜덤 | n_active=3 |
| ANCHOR | 문헌 조음 synergy + 교란 | 조음 영역 밀집 |
| EFFORT | effort×Dirichlet 코어 (주력 62%) | 희소↔밀집 **연속체** |
| LHS | 균일 LHS (OOD split용) | 거의 항상 11근 밀집 |

**핵심: 블록마다 분포가 완전히 다르므로 겹쳐 그리면 EFFORT(62%)가 전부 덮어버립니다.
→ 블록별로 나눠(facet) 보는 것이 분포 검증의 정답.** 아래는 그 관점의 표준 뷰 모음입니다.
가장 의미 있는 축은 생성기가 직접 제어하는 **effort × n_active** 예요 (③④).

In [ ]:
# --- ① 설정 & 임포트 ---
%matplotlib inline
from pathlib import Path
import csv
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

POOL = next((p for p in [Path('datasets/pool_v2_200000.txt'),
                         Path('test/datasets/pool_v2_200000.txt')] if p.exists()),
            Path('datasets/pool_v2_200000.txt'))
META = POOL.with_suffix('').with_suffix('.meta.csv')   # datasets/pool_v2_200000.meta.csv
# sampling 코드 경로 (meta.csv 가 없을 때 라벨 재생성용)
SAMPLING_DIR = next((p for p in [Path('sampling'), Path('test/sampling')] if p.exists()), None)

MUSCLE_NAMES = ['GGP','GGM','GGA','STY','GH','MH','HG','VERT','TRANS','IL','SL']
BLOCK_ORDER  = ['REST','SINGLE','PAIR','TRIPLE','ANCHOR','EFFORT','LHS']
BLOCK_COLORS = {'REST':'#000000','SINGLE':'#4C78A8','PAIR':'#F58518','TRIPLE':'#54A24B',
                'ANCHOR':'#E45756','EFFORT':'#B279A2','LHS':'#9D755D'}
N_SAMPLE = 20000; SEED = 0
print('POOL =', POOL.resolve())

In [ ]:
# --- ★ load_data(txt_path): TXT 로드 → 섹션(블록)별 인덱스 dict 반환 ---
import csv as _csv

def load_data(txt_path, meta_path=None, return_acts=False):
    """pool txt 를 읽어 각 sampling 섹션(블록)별 '행 인덱스'를 반환.

    반환: dict { 'rest':idx, 'single':idx, 'pair':idx, 'triple':idx,
                 'anchor':idx, 'effort':idx, 'lhs':idx }   (idx = np.ndarray[int])
        · 'pair' 가 그림의 'double' 입니다 (PAIR = 2근 격자).
    라벨 출처 우선순위: (0) txt 안의 'block' 열(태그드 파일) → (1) 옆의 *.meta.csv
                       → (2) sampling/build_pool.py 로 결정론적 재생성.
    return_acts=True 면 (idx_dict, acts) 튜플 반환.
    """
    txt_path = Path(txt_path)
    with open(txt_path) as _f:                              # 헤더(컬럼) 스니핑
        for _ in range(3): _f.readline()
        cols = _f.readline().strip().split(',')
    mcols = [k for k, c in enumerate(cols) if c in MUSCLE_NAMES]
    acts = np.loadtxt(txt_path, delimiter=',', skiprows=4, usecols=mcols)
    N = len(acts)

    block = None
    if 'block' in cols:                                     # (0) 태그드 txt (파일에 block 열)
        bi = cols.index('block')
        block = np.loadtxt(txt_path, delimiter=',', skiprows=4,
                           usecols=[bi], dtype=str).astype(object)
    if block is None:                                       # (1) 옆의 meta.csv
        mp = Path(meta_path) if meta_path else txt_path.parent / (txt_path.stem + '.meta.csv')
        if mp.exists():
            block = np.empty(N, object)
            with open(mp, newline='') as f:
                for i, row in enumerate(_csv.DictReader(f)):
                    block[i] = row['block']
    if block is None:                                       # (2) build_pool 로 결정론적 재생성
        for cand in [Path('sampling'), Path('test/sampling'),
                     txt_path.parent.parent / 'sampling']:
            if (cand / 'build_pool.py').exists():
                import sys; sys.path.insert(0, str(cand))
                import build_pool as bp
                A2, tags = bp.build(200000, 1.0, 20260714)
                if np.abs(np.round(A2, 6) - acts).max() < 1e-6:
                    block = np.asarray(tags, object)
                break
    if block is None:
        raise FileNotFoundError('block 열 / meta.csv / build_pool 모두 없음 — 라벨 생성 불가')

    order = ['REST','SINGLE','PAIR','TRIPLE','ANCHOR','EFFORT','LHS']
    idx = {b.lower(): np.flatnonzero(block == b) for b in order if (block == b).any()}
    return (idx, acts) if return_acts else idx


# 사용 예시 — 태그드 파일이 있으면 그걸(=meta 불필요), 없으면 원본 pool 사용
TAGGED = POOL.parent / 'pool_v2_200000_tagged.txt'
SRC = TAGGED if TAGGED.exists() else POOL
idx = load_data(SRC)
print('source =', SRC.name)
print('섹션별 인덱스:')
for name, ix in idx.items():
    head = ix[:3].tolist()
    print(f'  {name:8s} {len(ix):7d}  예) {head} … 마지막={int(ix[-1])}')

# 블록만 뽑아 쓰는 법:  acts[idx['single']],  acts[idx['effort']] …

In [ ]:
# --- ② pool + 블록 라벨 로드 ---
raw  = np.loadtxt(POOL, delimiter=',', skiprows=4)
acts = raw[:, 1:]
N    = len(acts)

def load_meta(path):
    blk = np.empty(N, object); eff = np.zeros(N); na = np.zeros(N, int)
    with open(path, newline='') as f:
        for i, row in enumerate(csv.DictReader(f)):
            blk[i] = row['block']; eff[i] = float(row['effort']); na[i] = int(row['n_active'])
    return blk, eff, na

if META.exists():
    block, effort, n_active = load_meta(META)
    print('meta.csv 로드:', META.name)
elif SAMPLING_DIR is not None:
    # meta.csv 가 없으면 동일 seed 로 라벨 재생성 (생성이 결정론적)
    import sys; sys.path.insert(0, str(SAMPLING_DIR))
    import build_pool as bp
    A2, tags = bp.build(200000, 1.0, 20260714)
    assert np.abs(np.round(A2,6) - acts).max() < 1e-6, 'pool 불일치 — seed/버전 확인'
    block = np.asarray(tags, object); effort = acts.sum(1); n_active = (acts > 0.05).sum(1)
    print('meta.csv 없음 → build_pool 로 라벨 재생성 완료')
else:
    block = np.full(N, 'ALL', object); effort = acts.sum(1); n_active = (acts > 0.05).sum(1)
    print('경고: 블록 라벨 원본 없음 → 단일 그룹으로 처리')

print(f'{N:,} 샘플 × {acts.shape[1]} 근육')

In [ ]:
# --- ③ 블록 구성 + 요약 통계 ---
present = [b for b in BLOCK_ORDER if (block == b).any()]
print(f'{"block":8s}{"count":>9s}{"%":>7s}{"effort(mean±sd)":>20s}{"n_active(mean)":>16s}')
for b in present:
    m = block == b; k = m.sum()
    print(f'{b:8s}{k:9d}{100*k/N:7.2f}{effort[m].mean():11.2f}±{effort[m].std():5.2f}{n_active[m].mean():16.2f}')

fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].bar(present, [ (block==b).sum() for b in present ],
          color=[BLOCK_COLORS[b] for b in present])
ax[0].set_yscale('log'); ax[0].set_ylabel('count (log)'); ax[0].set_title('Block composition')
for b in present:
    ax[1].hist(effort[block==b], bins=60, histtype='step', lw=1.6,
               color=BLOCK_COLORS[b], label=b, density=True)
ax[1].set_xlabel('effort = Σ activation'); ax[1].set_ylabel('density')
ax[1].set_title('Effort distribution by block'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# --- ④ [핵심] effort × n_active 를 블록별로 facet ---
# 생성기가 제어하는 두 축. 블록별로 어느 영역을 덮는지 한눈에 보입니다.
present = [b for b in BLOCK_ORDER if (block == b).any()]
ncol = 4; nrow = int(np.ceil(len(present)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4*ncol, 3.4*nrow), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()
for a, b in zip(axes, present):
    m = block == b
    a.hist2d(effort[m], n_active[m], bins=[40, 12],
             range=[[0, effort.max()], [0, 11.5]], cmap='viridis', cmin=1)
    a.set_title(f'{b}  (n={m.sum():,})', color=BLOCK_COLORS[b], fontsize=10)
    a.set_xlabel('effort'); a.set_ylabel('# active')
for a in axes[len(present):]: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# --- ⑤ 근육별 커버리지 (각 근육이 0..1 을 고루 덮는가) ---
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
# (a) 근육별 값 분포 violin (0 제외 = 실제 켜졌을 때의 세기)
data = [acts[acts[:, i] > 1e-6, i] for i in range(len(MUSCLE_NAMES))]
parts = ax[0].violinplot(data, showmedians=True)
ax[0].set_xticks(range(1, len(MUSCLE_NAMES)+1)); ax[0].set_xticklabels(MUSCLE_NAMES, rotation=45)
ax[0].set_ylabel('activation (when active)'); ax[0].set_title('Per-muscle intensity (nonzero)')
# (b) 근육별 '켜진 비율'
frac_on = (acts > 0.05).mean(0)
ax[1].bar(MUSCLE_NAMES, frac_on, color='#4C78A8')
ax[1].set_ylabel('fraction of samples active (>0.05)')
ax[1].set_title('Per-muscle activation frequency'); ax[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# --- ⑥ 11D PCA — 블록별 overlay & facet ---
rng = np.random.default_rng(SEED)
sub = rng.choice(N, size=min(30000, N), replace=False)
Xc = acts[sub] - acts[sub].mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
PC = Xc @ Vt[:2].T
bsub = block[sub]
present = [b for b in BLOCK_ORDER if (bsub == b).any()]

fig = plt.figure(figsize=(15, 6))
axO = fig.add_subplot(1, 2, 1)
for b in present:
    m = bsub == b
    axO.scatter(PC[m,0], PC[m,1], s=4, alpha=0.35, color=BLOCK_COLORS[b], label=b, linewidths=0)
axO.set_xlabel('PC1'); axO.set_ylabel('PC2'); axO.set_title('PCA overlay (color=block)')
axO.legend(fontsize=8, markerscale=2)
# facet: 각 블록만 강조
gs = fig.add_gridspec(2, 4, left=0.52, right=0.99, hspace=0.4, wspace=0.3)
for ax_i, b in zip([fig.add_subplot(gs[k//4, k%4]) for k in range(len(present))], present):
    ax_i.scatter(PC[:,0], PC[:,1], s=2, color='#DDDDDD', linewidths=0)
    m = bsub == b
    ax_i.scatter(PC[m,0], PC[m,1], s=3, color=BLOCK_COLORS[b], linewidths=0)
    ax_i.set_title(b, fontsize=9, color=BLOCK_COLORS[b]); ax_i.set_xticks([]); ax_i.set_yticks([])
print('PC 설명분산:', ((S**2)/(S**2).sum())[:2].round(3))
plt.show()

In [ ]:
# --- ⑦ (원래 목표) 3그룹 축 산점도 — 이번엔 '진짜 블록' 색으로 ---
from mpl_toolkits.mplot3d import Axes3D  # noqa
GROUPS = {'G1 Genioglossus':['GGP','GGM','GGA'],
          'G2 Extrinsic/floor':['STY','GH','MH','HG'],
          'G3 Intrinsic':['VERT','TRANS','IL','SL']}
col = {m:i for i,m in enumerate(MUSCLE_NAMES)}
gn = list(GROUPS)
G = np.stack([acts[:,[col[m] for m in GROUPS[g]]].mean(1) for g in gn], 1)
sel = rng.choice(N, size=min(N_SAMPLE, N), replace=False)
fig = plt.figure(figsize=(9,8)); ax = fig.add_subplot(111, projection='3d')
for b in [b for b in BLOCK_ORDER if (block[sel]==b).any()]:
    m = block[sel]==b
    ax.scatter(G[sel][m,0], G[sel][m,1], G[sel][m,2], s=5, alpha=0.4,
               color=BLOCK_COLORS[b], label=b, linewidths=0)
ax.set_xlabel(gn[0]); ax.set_ylabel(gn[1]); ax.set_zlabel(gn[2])
ax.set_title('3-group means, colored by real block'); ax.legend(fontsize=8, markerscale=2)
ax.view_init(elev=20, azim=45); plt.tight_layout(); plt.show()

In [ ]:
# --- ⑧ (선택) plotly 인터랙티브 3D — 블록별 토글 ---
try:
    import plotly.graph_objects as go
except ModuleNotFoundError:
    import sys, subprocess; subprocess.run([sys.executable,'-m','pip','install','-q','plotly'], check=True)
    import plotly.graph_objects as go
fig = go.Figure()
for b in [b for b in BLOCK_ORDER if (block[sel]==b).any()]:
    m = block[sel]==b
    fig.add_trace(go.Scatter3d(x=G[sel][m,0], y=G[sel][m,1], z=G[sel][m,2], mode='markers',
                  name=b, marker=dict(size=2, color=BLOCK_COLORS[b], opacity=0.5)))
fig.update_layout(scene=dict(xaxis_title=gn[0], yaxis_title=gn[1], zaxis_title=gn[2]),
                  title='3-group means by block (범례 클릭으로 토글)', width=820, height=720)
fig.show()

In [ ]:
# --- ⑨ ANCHOR 중심별 분리도 (출처 anchor 재구성 + PCA + purity) ---
# ANCHOR 블록의 각 샘플이 '어느 중심에서 나왔는지'를 build_pool 로 결정론적 재구성해 색칠.
# 데이터가 잘 구별되면 색 덩어리가 분리되고 purity(최근접-중심 정확도)가 높습니다.
import sys, importlib
SAMP = next((p for p in [Path('sampling'), Path('test/sampling')]
             if (p/'build_pool.py').exists()), None)
assert SAMP is not None, 'sampling/build_pool.py 를 찾을 수 없습니다'
sys.path.insert(0, str(SAMP))
import muscles as M, build_pool as bp
importlib.reload(M); importlib.reload(bp)

SEED, CAP, NTOT = 20260714, 1.0, 200000
ANCHOR_WIDE = False   # pool 을 anchor_wide=True 로 만들었으면 여기도 True

def build_with_src(n_total, cap, seed, anchor_wide):
    rng = np.random.default_rng(seed)
    names = list(M.ARTICULATORY_ANCHORS)
    base = np.stack([M.anchor_to_vec(M.ARTICULATORY_ANCHORS[n], cap) for n in names])
    fixed = [bp.block_rest(), bp.block_single(cap), bp.block_pair(cap)]
    n_fixed = sum(b[0].shape[0] for b in fixed)
    n_rand = max(0, n_total - n_fixed)
    fe,fa,ft,fl = 0.62,0.14,0.06,0.18; s=fe+fa+ft+fl
    k_anc=int(round(n_rand*fa/s)); k_tri=int(round(n_rand*ft/s))
    k_lhs=int(round(n_rand*fl/s)); k_eff=n_rand-k_anc-k_tri-k_lhs
    eff = bp.block_effort(rng, k_eff, cap)                      # rng 순서 동일
    kw = dict(sigma=0.12,pd=0.15,pe=0.25,elo=0.4,ehi=1.8) if anchor_wide \
         else dict(sigma=0.05,pd=0.0,pe=0.0,elo=0.7,ehi=1.3)
    rows=np.zeros((k_anc,M.N_MUSCLES)); src=np.empty(k_anc,int)
    for r in range(k_anc):
        si=rng.integers(len(names)); src[r]=si; b=base[si].copy()
        b*=rng.uniform(kw['elo'],kw['ehi']); b+=rng.normal(0,kw['sigma'],M.N_MUSCLES)*(b>0)
        act=np.flatnonzero(b>1e-6)
        if len(act)>1 and rng.random()<kw['pd']: b[rng.choice(act)]=0.0
        if rng.random()<kw['pe']:
            offv=np.flatnonzero(b<=1e-6)
            if len(offv): b[rng.choice(offv)]=rng.uniform(0.05,0.45)
        rows[r]=np.clip(b,0,cap)
    tri=bp.block_triple(rng,k_tri,cap); lhs=bp.block_lhs(rng,k_lhs,cap)
    rand=[eff,(rows,'ANCHOR'),tri,lhs]
    Ar=np.concatenate([b[0] for b in rand])
    src_full=np.full(Ar.shape[0],-1,int); src_full[k_eff:k_eff+k_anc]=src
    order=rng.permutation(Ar.shape[0])
    A=np.concatenate([b[0] for b in fixed]+[Ar[order]])
    asrc=np.concatenate([np.full(n_fixed,-1,int), src_full[order]])
    return A, np.asarray(names), asrc

Are, anames, asrc = build_with_src(NTOT, CAP, SEED, ANCHOR_WIDE)
print('build_pool 재현 일치:', np.abs(Are - bp.build(NTOT,CAP,SEED,anchor_wide=ANCHOR_WIDE)[0]).max()==0)
print('중심(anchor) 개수:', len(anames))

# ANCHOR 블록만 추출
am = asrc >= 0
X = Are[am]; lab = asrc[am]
base = np.stack([M.anchor_to_vec(M.ARTICULATORY_ANCHORS[n], CAP) for n in anames])
def _l1(Z): s=Z.sum(1,keepdims=True); s[s==0]=1; return Z/s
pred = np.linalg.norm(_l1(X)[:,None]-_l1(base)[None],axis=2).argmin(1)
purity = (pred==lab).mean()

# PCA 2D
mu=X.mean(0); U,S,Vt=np.linalg.svd(X-mu,full_matrices=False)
P=(X-mu)@Vt[:2].T; Cp=(base-mu)@Vt[:2].T
colors = plt.cm.gist_ncar(np.linspace(0,1,len(anames)))
fig, ax = plt.subplots(figsize=(11,8))
for i in range(len(anames)):
    m = lab==i
    if m.any(): ax.scatter(P[m,0],P[m,1],s=6,alpha=0.5,color=colors[i],linewidths=0,label=anames[i])
ax.scatter(Cp[:,0],Cp[:,1],c='k',marker='*',s=90,zorder=5)
ax.set_title(f'ANCHOR per-center separability — purity={100*purity:.1f}%  '
             f'({len(anames)} centers, {"wide" if ANCHOR_WIDE else "tight"})')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(fontsize=6, ncol=2, loc='center left', bbox_to_anchor=(1.0,0.5))
plt.tight_layout(); plt.show()